<a href="https://colab.research.google.com/github/sarmad-usman/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarmad-usman/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [8]:
import pandas as pd
import numpy as np

# --- 1. My rule and its reason codes ---
# The rule: Assign a baseline action score based on a 'user_activity_score'.
# Users with high activity get a higher score.

# Define reason codes
REASON_CODE_HIGH_ACTIVITY = "R01: High recent activity"
REASON_CODE_MEDIUM_ACTIVITY = "R02: Medium recent activity"
REASON_CODE_LOW_ACTIVITY = "R03: Low recent activity"

# Create a dummy DataFrame to demonstrate the rule
data = {
    'user_id': range(1, 11),
    'user_activity_score': [85, 20, 60, 95, 30, 70, 10, 55, 40, 80],
    'days_since_last_purchase': [5, 30, 15, 2, 60, 10, 90, 20, 45, 7]
}
df = pd.DataFrame(data)

def apply_baseline_rule(row):
    score = 0
    reason_code = ""

    activity = row['user_activity_score']

    if activity >= 75:
        score = 0.9
        reason_code = REASON_CODE_HIGH_ACTIVITY
    elif activity >= 50:
        score = 0.6
        reason_code = REASON_CODE_MEDIUM_ACTIVITY
    else:
        score = 0.3
        reason_code = REASON_CODE_LOW_ACTIVITY

    return pd.Series({'baseline_action_score': score, 'reason_code': reason_code})

# Apply the rule to the DataFrame
df[['baseline_action_score', 'reason_code']] = df.apply(apply_baseline_rule, axis=1)

# Display the results for a few users
print("Example of baseline action scores and reason codes:")
print(df[['user_id', 'user_activity_score', 'baseline_action_score', 'reason_code']].head())

# You can also print the unique reason codes defined
print("\nDefined Reason Codes:")
print(f"- {REASON_CODE_HIGH_ACTIVITY}")
print(f"- {REASON_CODE_MEDIUM_ACTIVITY}")
print(f"- {REASON_CODE_LOW_ACTIVITY}")

Example of baseline action scores and reason codes:
   user_id  user_activity_score  baseline_action_score  \
0        1                   85                    0.9   
1        2                   20                    0.3   
2        3                   60                    0.6   
3        4                   95                    0.9   
4        5                   30                    0.3   

                   reason_code  
0    R01: High recent activity  
1     R03: Low recent activity  
2  R02: Medium recent activity  
3    R01: High recent activity  
4     R03: Low recent activity  

Defined Reason Codes:
- R01: High recent activity
- R02: Medium recent activity
- R03: Low recent activity


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
import os

# Sort the DataFrame by baseline_action_score in descending order and add a rank
ranked_df = df.sort_values(by='baseline_action_score', ascending=False).reset_index(drop=True)
ranked_df['rank'] = ranked_df.index + 1

# Select the relevant columns for the output
output_columns = ['rank', 'user_id', 'baseline_action_score', 'reason_code', 'user_activity_score', 'days_since_last_purchase']
final_output_df = ranked_df[output_columns]

# Define the output directory and filename
output_dir = 'work/outputs'
output_filename = os.path.join(output_dir, 'baseline_action_score.csv')

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Write the DataFrame to a CSV file
final_output_df.to_csv(output_filename, index=False)

print(f"Ranked queue successfully written to {output_filename}")

# Display the top 10 ranked users
print("\nTop 10 ranked users:")
print(final_output_df.head(10))

Ranked queue successfully written to work/outputs/baseline_action_score.csv

Top 10 ranked users:
   rank  user_id  baseline_action_score                  reason_code  \
0     1        1                    0.9    R01: High recent activity   
1     2        4                    0.9    R01: High recent activity   
2     3       10                    0.9    R01: High recent activity   
3     4        8                    0.6  R02: Medium recent activity   
4     5        6                    0.6  R02: Medium recent activity   
5     6        3                    0.6  R02: Medium recent activity   
6     7        2                    0.3     R03: Low recent activity   
7     8        5                    0.3     R03: Low recent activity   
8     9        7                    0.3     R03: Low recent activity   
9    10        9                    0.3     R03: Low recent activity   

   user_activity_score  days_since_last_purchase  
0                   85                         5  
1      

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
import pandas as pd
import os

# Define the path to the previously generated CSV
output_dir = 'work/outputs'
output_filename = os.path.join(output_dir, 'baseline_action_score.csv')

try:
    # Load the ranked queue
    ranked_queue = pd.read_csv(output_filename)

    # Get the top 20 users (or fewer if the dataset is smaller)
    top_20_users = ranked_queue.head(20)

    print("--- Top 20 Review ---")
    print("Reviewing the top users identified by the baseline action score.\n")

    if top_20_users.empty:
        print("No users found in the ranked queue to review.")
    else:
        for index, row in top_20_users.iterrows():
            user_id = row['user_id']
            baseline_score = row['baseline_action_score']
            reason_code = row['reason_code']
            user_activity_score = row['user_activity_score']
            days_since_last_purchase = row['days_since_last_purchase']
            rank = row['rank']

            action = ""
            confidence_note = ""
            what_would_make_it_wrong = ""

            # Interpret the baseline score and provide a review
            if baseline_score >= 0.7: # Corresponds to REASON_CODE_HIGH_ACTIVITY (0.9 in the dummy data)
                action = f"Target for immediate engagement campaign (e.g., personalized offer, exclusive content)."
                confidence_note = f"High confidence. User {user_id} has a high activity score ({user_activity_score}) suggesting strong recent engagement, making them a prime candidate. The score is directly tied to observed activity, a good proxy for current interest."
                what_would_make_it_wrong = "The activity might be superficial (e.g., browsing but not converting), or the 'activity score' might be influenced by non-human interaction (e.g., bots)."
            elif baseline_score >= 0.5: # Corresponds to REASON_CODE_MEDIUM_ACTIVITY (0.6 in the dummy data)
                action = f"Consider for re-engagement or loyalty program outreach. Monitor for increased activity."
                confidence_note = f"Medium confidence. User {user_id} shows decent activity ({user_activity_score}), indicating potential. A well-timed intervention could convert them, as the rule captures moderate engagement."
                what_would_make_it_wrong = "Their activity might be declining, or their past purchasing behavior doesn't align with their current activity level. The 'medium activity' threshold might include users with varied true intent."
            else: # Corresponds to REASON_CODE_LOW_ACTIVITY (0.3 in the dummy data)
                action = f"Include in a broader awareness campaign or passively monitor for future activity. Not a priority for direct action."
                confidence_note = f"Lower confidence for direct action. User {user_id} has lower activity ({user_activity_score}). While they made it into the top list (possibly due to a small dataset or a generous rule), their engagement signal is weaker."
                what_would_make_it_wrong = "Their low activity could be an anomaly, or they might be a high-value customer who simply doesn't interact frequently. The rule might be missing other crucial signals for these types of users."

            print(f"--- Rank {rank}: User ID {user_id} (Baseline Score: {baseline_score:.2f}) ---")
            print(f"  User Activity Score: {user_activity_score}")
            print(f"  Days Since Last Purchase: {days_since_last_purchase}")
            print(f"  Action: {action}")
            print(f"  Reason Code: {reason_code}")
            print(f"  Confidence Note: {confidence_note}")
            print(f"  What would make it wrong: {what_would_make_it_wrong}\n")

except FileNotFoundError:
    print(f"Error: The file '{output_filename}' was not found. Please ensure the previous cell (Section 2) was run successfully to generate the CSV.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

--- Top 20 Review ---
Reviewing the top users identified by the baseline action score.

--- Rank 1: User ID 1 (Baseline Score: 0.90) ---
  User Activity Score: 85
  Days Since Last Purchase: 5
  Action: Target for immediate engagement campaign (e.g., personalized offer, exclusive content).
  Reason Code: R01: High recent activity
  Confidence Note: High confidence. User 1 has a high activity score (85) suggesting strong recent engagement, making them a prime candidate. The score is directly tied to observed activity, a good proxy for current interest.
  What would make it wrong: The activity might be superficial (e.g., browsing but not converting), or the 'activity score' might be influenced by non-human interaction (e.g., bots).

--- Rank 2: User ID 4 (Baseline Score: 0.90) ---
  User Activity Score: 95
  Days Since Last Purchase: 2
  Action: Target for immediate engagement campaign (e.g., personalized offer, exclusive content).
  Reason Code: R01: High recent activity
  Confidence No

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [11]:
import pandas as pd
import os

# Define the path to the previously generated CSV
output_dir = 'work/outputs'
output_filename = os.path.join(output_dir, 'baseline_action_score.csv')

try:
    # Load the ranked queue to perform checks
    ranked_queue = pd.read_csv(output_filename)

    print("--- 4. Weak Picks + Leakage Check ---")
    print("Reviewing the ranked queue for potential 'weak' picks and ensuring no data leakage.\n")

    if ranked_queue.empty:
        print("No ranked queue found to analyze for weak picks or leakage.")
    else:
        # Identify 'weak picks': This is subjective and depends on campaign goals.
        # For this simple rule, a 'weak pick' might be a highly active user who *just* made a purchase,
        # meaning they might not need an *immediate re-engagement* action despite a high score.
        # Or, a low-scoring user that might be valuable but the rule misses them.

        print("### Weak Picks Analysis ###")

        # Example 1: High score, but very recent purchase (might not need immediate 'action' for re-engagement)
        weak_pick_candidates = ranked_queue[
            (ranked_queue['baseline_action_score'] >= 0.7) &
            (ranked_queue['days_since_last_purchase'] <= 7)
        ]

        if not weak_pick_candidates.empty:
            print("Some users have high baseline scores AND very recent purchases (<= 7 days). While high-value, they might not need *immediate re-engagement* actions as they are already active/engaged. They might be 'weak picks' if the action is specifically about reactivating dormant users.")
            print("Considered 'Weak Picks' for *re-engagement* campaigns (high score, recent purchase):")
            print(weak_pick_candidates[['user_id', 'baseline_action_score', 'user_activity_score', 'days_since_last_purchase']])
            print("\n")
        else:
            print("No users identified as 'weak picks' based on high score and very recent purchase within the dummy data.")
            print("\n")

        # Example 2: Low activity score, but maybe historically valuable (not covered by current data)
        # Since we only have activity and last purchase, we can't definitively say a low-score user is a 'weak pick' to ignore.
        # However, it's worth noting that the current rule might miss historically valuable but temporarily inactive users.
        low_activity_picks = ranked_queue[ranked_queue['baseline_action_score'] == 0.3]
        if not low_activity_picks.empty:
            print("Users with low activity scores (0.3) are given a lower priority. These might be 'weak picks' if the goal is to identify ALL potential high-value users, including those who are currently inactive but historically valuable. Our current rule, solely based on 'user_activity_score', does not capture historical value.")
            print("Example Low Activity Picks:")
            print(low_activity_picks.head(3)[['user_id', 'baseline_action_score', 'user_activity_score', 'days_since_last_purchase']])
            print("\n")

        print("### Leakage Check ###")
        print("The current rule for `baseline_action_score` is based on `user_activity_score`. This score represents the user's current or recent activity levels. We confirm the following:")
        print("- **No future window data leaked in**: The `user_activity_score` is assumed to be a metric calculated up to the point of scoring, not including any future activity or purchase data.")
        print("- **No product flags leaked in**: The rule does not incorporate any 'product flags' or internal states that would not be generally available or could inadvertently predict future outcomes that the model is trying to influence. It relies solely on the provided activity score.")
        print("- **No direct outcome leakage**: The rule uses `user_activity_score` and `days_since_last_purchase` (for context in review) which are input features, not the target outcome (e.g., whether a user *will* respond to a campaign). The rule generates a *propensity* score based on observed features.")
        print("This ensures the baseline score is a fair assessment based on available, non-leaky data.")

except FileNotFoundError:
    print(f"Error: The file '{output_filename}' was not found. Please ensure the previous cell (Section 2) was run successfully to generate the CSV.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

--- 4. Weak Picks + Leakage Check ---
Reviewing the ranked queue for potential 'weak' picks and ensuring no data leakage.

### Weak Picks Analysis ###
Some users have high baseline scores AND very recent purchases (<= 7 days). While high-value, they might not need *immediate re-engagement* actions as they are already active/engaged. They might be 'weak picks' if the action is specifically about reactivating dormant users.
Considered 'Weak Picks' for *re-engagement* campaigns (high score, recent purchase):
   user_id  baseline_action_score  user_activity_score  \
0        1                    0.9                   85   
1        4                    0.9                   95   
2       10                    0.9                   80   

   days_since_last_purchase  
0                         5  
1                         2  
2                         7  


Users with low activity scores (0.3) are given a lower priority. These might be 'weak picks' if the goal is to identify ALL potential 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.